In [8]:
from transformers import T5TokenizerFast, T5ForConditionalGeneration
import torch
import pandas as pd

In [1]:
from utils.config import *

In [10]:
dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'


#qa model
qa_type_model_name= 't5-base'
qa_type_model_result= '.temp/model_results/fine_tuned_question_answer_model-base'
qa_type_model= '.temp/model/fine_tuned_question_answer_model-base'



In [5]:
# question= "What is your name?"

In [6]:
# # Tokenize input and move tensors to the same device as the model
# inputs = QUESTION_CLASSIFER_TOKENIZER(question, return_tensors="pt", truncation=True, padding=True)
# inputs = {k: v.to(MOCEL_DEVICE) for k, v in inputs.items()}  # Move input tensors to same device
# # Perform inference
# with torch.no_grad():  # Disable gradient calculation for faster inference
#     outputs = QUESTION_CLASSIFER_MODEL(**inputs)

# # Extract logits and get the predicted class index
# logits = outputs.logits
# predicted_class = torch.argmax(logits, dim=1).item()
# ID2LABEL.get(predicted_class, "Unknown Class")

'personal_information'

### Preprocessing

In [9]:
df= pd.read_csv(dataset_path)
df= df[["question", "question_type"]]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1603 entries, 0 to 1602
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1603 non-null   object
 1   question_type  1603 non-null   object
dtypes: object(2)
memory usage: 25.2+ KB


In [11]:
tokenizer = T5TokenizerFast.from_pretrained(qa_type_model_name)
model = T5ForConditionalGeneration.from_pretrained(qa_type_model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [20]:
# Example question and a very long context (more than 800 tokens)
question = "What is your expected ctc in LPA"
context = "My expected cost to company (CTC) is ₹850000 per annum, which translates to an annual package of 8.5 LPA. When broken down on a monthly basis, this amounts to approximately ₹60,000 INR. If converted to USD, based on an approximate exchange rate of 1 USD = 83 INR, my expected annual earnings would be around $10,240, while my monthly salary would be approximately $723. This compensation expectation is based on my industry experience, technical expertise, and the value I bring to an organization through my skills, dedication, and problem-solving abilities. Over time, I have gained substantial experience and honed my capabilities, making me well-equipped to take on more challenging roles and responsibilities. My professional journey has been defined by continuous learning, adaptability, and a results-driven approach, which has allowed me to make meaningful contributions to my current and past organizations\nHaving worked extensively in my domain, I have acquired a deep understanding of industry trends, best practices, and cutting-edge technologies, enabling me to perform efficiently and contribute to organizational growth. My expected CTC is aligned with industry standards for professionals with similar experience, skill sets, and job responsibilities. It not only reflects my qualifications and expertise but also acknowledges my commitment to excellence and my ability to drive innovation and efficiency within a team. While compensation is an important factor in career progression, I also value opportunities for learning, career advancement, and exposure to new challenges that contribute to my overall professional growth.\nAdditionally, while my expected salary is set based on a fair assessment of my skills, experience, and market trends, I am open to negotiation depending on the overall compensation structure, benefits, incentives, and career development opportunities offered by the company. Factors such as performance-based bonuses, stock options, flexible work arrangements, learning opportunities, health benefits, and other perks also play a significant role in determining the overall attractiveness of a compensation package. I believe in a holistic approach to career decisions, where financial growth is complemented by professional development and work-life balance. Therefore, I am willing to discuss and explore opportunities that not only offer competitive remuneration but also align with my long-term career aspirations, allowing me to make a significant impact within the organization."  # A large document that exceeds 800 tokens

# Prepare the input
input_text = f"question: {question} context: {context}"

# Tokenize input (allow longer context, e.g., up to 1024 tokens if using t5-large)
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True, max_length=1024)

# Move input tensors to the same device as the model
inputs = {key: value.to(device) for key, value in inputs.items()}


In [21]:
# Perform inference (disable gradient calculation)
with torch.no_grad():
    outputs = model.generate(**inputs)

# Decode the generated tokens to get the answer
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Answer: {answer}")


Answer: my expected cost to company (CTC) is 850000 per annum, which 
